# Hooks Part 2: Post-Tool-Use Guardrails

Post-tool-use hooks run **after** a tool executes but before its result reaches the model — useful for validating or scrubbing results, like redacting a field that should never leave your backend.


In [1]:
from typing import Any

from claude_agent_sdk import (
    tool,  # decorator that turns a Python function into a Claude-usable tool
    create_sdk_mcp_server,  # bundles one or more tools into a "server" Claude can talk to
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    HookMatcher,  # says WHICH tool a hook should watch
    HookContext,  # extra info passed into a hook function when it fires
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)

MOCK_PRICES = {"AAPL": 193.50, "GOOGL": 178.25, "MSFT": 412.80}


def get_stock_price(ticker: str) -> dict[str, Any]:
    """Mock stock price lookup that also carries an internal-only field."""
    # internal_notes simulates a sensitive field a backend might accidentally
    # include in a tool's response — something that should never reach Claude.
    return {
        "price": MOCK_PRICES.get(ticker.upper(), 100.00),
        "internal_notes": "Sourced from internal trading desk feed #472 — do not expose to end users.",
    }


@tool("get_stock_price", "Get the current mock stock price for a ticker symbol", {"ticker": str})
async def get_stock_price_tool(args: dict[str, Any]) -> dict[str, Any]:
    data = get_stock_price(args["ticker"])
    text = f"{args['ticker']}: ${data['price']:.2f}\nInternal notes: {data['internal_notes']}"
    return {"content": [{"type": "text", "text": text}]}


stock_server = create_sdk_mcp_server(name="stocks", version="1.0.0", tools=[get_stock_price_tool])

## A `PostToolUse` hook that strips `internal_notes` before Claude sees it


In [2]:
# This is a "PostToolUse" hook — it runs automatically right AFTER a tool
# finishes, but BEFORE Claude gets to see the result. That's our chance to
# clean up / redact the output before the model ever reads it.
async def redact_internal_notes(
    input_data: dict[str, Any],  # includes which hook event this is + the tool's raw result
    tool_use_id: str | None,  # unique id for this specific tool call
    context: HookContext,  # extra session context (not used here)
) -> dict[str, Any]:
    if input_data["hook_event_name"] != "PostToolUse":
        return {}

    tool_response = input_data["tool_response"]
    # tool_response has shown up as either {"content": [...]} or a bare content list
    content_blocks = tool_response["content"] if isinstance(tool_response, dict) else tool_response
    raw_content = content_blocks[0]["text"]
    # Cut off everything from "\nInternal notes:" onward — that's the redaction.
    redacted_content = raw_content.split("\nInternal notes:")[0]

    print(f"[hook] raw tool output:      {raw_content!r}")
    print(f"[hook] redacted tool output: {redacted_content!r}")

    # Returning "updatedToolOutput" tells the SDK: "replace the tool's real
    # result with THIS text before passing it on to Claude."
    return {
        "hookSpecificOutput": {
            "hookEventName": "PostToolUse",
            "updatedToolOutput": [{"type": "text", "text": redacted_content}],
        }
    }


options = ClaudeAgentOptions(
    model="haiku",
    mcp_servers={"stocks": stock_server},
    allowed_tools=["mcp__stocks__get_stock_price"],
    # hooks: this time registered under "PostToolUse" instead of "PreToolUse" —
    # same idea, just fires after the tool runs instead of before.
    hooks={"PostToolUse": [HookMatcher(matcher="mcp__stocks__get_stock_price", hooks=[redact_internal_notes])]},
)

## Run it — Claude never sees `internal_notes`


In [3]:
async def run_with_redaction() -> None:
    async with ClaudeSDKClient(options=options) as client:
        # We explicitly ask Claude to "repeat back everything the tool told you"
        # to prove the redaction really worked — if internal_notes leaked
        # through, we'd see it quoted back here.
        await client.query("What's the price of AAPL? Repeat back everything the tool told you.")
        async for message in client.receive_response():
            if isinstance(message, ResultMessage):
                print(f"\n[Claude's final answer] {message.result}")


await run_with_redaction()

[hook] raw tool output:      'AAPL: $193.50\nInternal notes: Sourced from internal trading desk feed #472 — do not expose to end users.'
[hook] redacted tool output: 'AAPL: $193.50'

[Claude's final answer] Here's what the tool returned:

**AAPL: $193.50**


## Summary

- A `PostToolUse` hook can rewrite a tool's result via `updatedToolOutput` — the model only ever sees what survives the hook.
- This is a real guardrail, not a convention: even a tool built to leak `internal_notes` can't get it past this hook.
